# Gold Layer — Orchestration
Run all gold notebooks in sequence. Single entry point for the Gold layer.

## Setup Connection

In [1]:
import os
from dotenv import load_dotenv
from clickzetta.zettapark.session import Session
from clickzetta.zettapark import functions as F
from clickzetta.zettapark.window import Window

load_dotenv()
session = Session.builder.configs({
    "username":  os.environ["CLICKZETTA_USERNAME"],
    "password":  os.environ["CLICKZETTA_PASSWORD"],
    "service":   os.environ["CLICKZETTA_SERVICE"],
    "instance":  os.environ["CLICKZETTA_INSTANCE"],
    "workspace": os.environ["CLICKZETTA_WORKSPACE"],
    "schema":    os.environ["CLICKZETTA_SCHEMA"],
    "vcluster":  os.environ["CLICKZETTA_VCLUSTER"],
}).create()
SCHEMA = os.environ["CLICKZETTA_SCHEMA"]

## Orchestration Logic
Import and run each gold function directly — no subprocess needed.

In [2]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))

from clickzetta.zettapark import functions as F
from clickzetta.zettapark.window import Window

### dim_customers

In [3]:
# ── dim_customers ──────────────────────────────────────────────────────────
ci = session.table(f"{SCHEMA}.crm_customers")
ca = session.table(f"{SCHEMA}.erp_customers")
la = session.table(f"{SCHEMA}.erp_customer_location")

joined = (ci.join(ca, ci["customer_number"] == ca["customer_number"], "left")
            .join(la, ci["customer_number"] == la["customer_number"], "left"))
df = joined.select(
    ci["customer_id"].alias("customer_id"),
    ci["customer_number"].alias("customer_number"),
    ci["first_name"].alias("first_name"),
    ci["last_name"].alias("last_name"),
    la["country"].alias("country"),
    ci["marital_status"].alias("marital_status"),
    F.when(ci["gender"] != "n/a", ci["gender"])
     .otherwise(F.coalesce(ca["gender"], F.lit("n/a"))).alias("gender"),
    ca["birth_date"].alias("birthdate"),
    ci["created_date"].alias("create_date"),
)
w = Window.order_by(F.col("customer_id"))
df = df.with_column("customer_key", F.row_number().over(w))
df.select("customer_key","customer_id","customer_number","first_name","last_name",
          "country","marital_status","gender","birthdate","create_date") \
  .write.save_as_table(f"{SCHEMA}.dim_customers", mode="overwrite")
print("dim_customers OK")

dim_customers OK


### dim_products

In [4]:
# ── dim_products ──────────────────────────────────────────────────────────
pn = session.table(f"{SCHEMA}.crm_products")
pc = session.table(f"{SCHEMA}.erp_product_category")

joined = pn.join(pc, pn["category_id"] == pc["category_id"], "left")
df = joined.select(
    pn["product_id"].alias("product_id"),
    pn["product_number"].alias("product_number"),
    pn["product_name"].alias("product_name"),
    pn["category_id"].alias("category_id"),
    pc["category"].alias("category"),
    pc["subcategory"].alias("subcategory"),
    pc["maintenance_flag"].alias("maintenance_flag"),
    pn["product_line"].alias("product_line"),
    pn["start_date"].alias("start_date"),
)
w = Window.order_by(F.col("start_date"), F.col("product_number"))
df = df.with_column("product_key", F.row_number().over(w))
df.select("product_key","product_id","product_number","product_name",
          "category_id","category","subcategory","maintenance_flag","product_line","start_date") \
  .write.save_as_table(f"{SCHEMA}.dim_products", mode="overwrite")
print("dim_products OK")

dim_products OK


### fact_sales

In [5]:
# ── fact_sales ────────────────────────────────────────────────────────────
sd = session.table(f"{SCHEMA}.crm_sales")
pr = session.table(f"{SCHEMA}.dim_products")
cu = session.table(f"{SCHEMA}.dim_customers")

joined = (sd.join(pr, sd["product_number"] == pr["product_number"], "left")
            .join(cu, sd["customer_id"] == cu["customer_id"], "left"))
df = joined.select(
    sd["order_number"].alias("order_number"),
    pr["product_key"].alias("product_key"),
    cu["customer_key"].alias("customer_key"),
    sd["order_date"].alias("order_date"),
    sd["ship_date"].alias("ship_date"),
    sd["due_date"].alias("due_date"),
    sd["sales_amount"].alias("sales_amount"),
    sd["quantity"].alias("quantity"),
    sd["price"].alias("price"),
)
df.write.save_as_table(f"{SCHEMA}.fact_sales", mode="overwrite")
print("fact_sales OK")

fact_sales OK
